# MistSense — Walkthrough

Paper: Patsch et al., *MistSense: Versatile Online Detection of Procedural and Execution Mistakes*, ICCV 2025. [PDF](https://openaccess.thecvf.com/content/ICCV2025/papers/Patsch_MistSense_Versatile_Online_Detection_of_Procedural_and_Execution_Mistakes_ICCV_2025_paper.pdf)

This notebook connects each paper section to the corresponding code and runs minimal sanity checks. It does not train the model — for that use `src/train.py` after preparing the dataset.

Ambiguities and choices are documented in `REPRODUCTION_NOTES.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))  # add baseline/MistSense to sys.path

import torch
from src.model import MistSense, ModelConfig
from src.loss import WeightedBCELoss
from src.evaluate import detection_metrics
from src.utils import Config, count_trainable

torch.manual_seed(0)

## §3.1 — Visual branch shapes

Paper: `v = [v_{t-t_s}, ..., v_t]`, `v ∈ R^{t_s × d_1}`. With learnable queries `q_rgb ∈ R^{t_q × d_2}`, the Video Q-Former yields `f_rgb ∈ R^{t_q × d_2}`.

Defaults (see `REPRODUCTION_NOTES.md`): `t_s=8`, `d_1=1024`, `t_q=32`, `d_2=768`.

In [ ]:
cfg = ModelConfig()
model = MistSense(cfg)

B = 2
visual = torch.randn(B, cfg.segment_length, cfg.d_visual)
f_rgb = model.encode_video(visual)
assert f_rgb.shape == (B, cfg.num_queries, cfg.d_model), f_rgb.shape
print('f_rgb shape OK:', f_rgb.shape)

## §3.1 — Hand pose branch shapes

Paper: `h ∈ R^{c × t_s × v × m}` (coord × time × joints × hands). Per-joint TCN + self-attention → `h_1 ∈ R^{t_s × d_h}` → projection → `h_2 ∈ R^{t_s × d_2}` → Hand Q-Former → `f_hand ∈ R^{t_q × d_2}`.

Defaults: `c=3` (x,y,z), `v=21` joints (Mediapipe), `m=2` hands, `d_h=256`.

In [ ]:
hand = torch.randn(B, cfg.coord_dim, cfg.segment_length, cfg.num_joints, cfg.num_hands)
f_hand = model.encode_hand(hand)
assert f_hand.shape == (B, cfg.num_queries, cfg.d_model), f_hand.shape
print('f_hand shape OK:', f_hand.shape)

## §3.1 — Fusion + mistake head

Paper: "The features `f_rgb` and `f_hand` are then added together and passed to the mistake classification layer to obtain the final mistake logits `m ∈ R^{t_q}`."

In [ ]:
out = model(visual, hand)
print('logits:', out['logits'].shape)            # (B, t_q)
print('gate (max σ over queries):', out['gate']) # (B,)
print('explain?', model.should_explain(out['gate']))
assert out['logits'].shape == (B, cfg.num_queries)

## §3.2 — Gating and LLM projection

Paper: once `g_m = σ(m) ≥ τ`, the Video Q-Former features `f_rgb` are routed through a linear projection to the LLM embedding space (Llama2-7B, hidden = 4096).

[UNSPECIFIED] Paper gives no value of τ — default 0.5 (configurable).

In [ ]:
proj = model.project_to_llm(out['f_rgb'])
print('LLM input tokens shape:', proj.shape)     # (B, t_q, 4096)
assert proj.shape == (B, cfg.num_queries, cfg.llm_hidden)

## §3.3 — Weighted BCE loss

Paper equation (rendered as image in the PDF):
$$L_{det} = -\frac{1}{N}\sum_i \left[ w_1\, y_i \log \hat{y}_i + w_0\, (1 - y_i)\log (1 - \hat{y}_i) \right]$$

Class weights (§4.6): Epic-tent w_0=w_1=1; HoloAssist w_0=1, w_1=4.

In [ ]:
loss_fn = WeightedBCELoss(w0=1.0, w1=4.0)        # HoloAssist setting
target = (torch.rand_like(out['logits']) > 0.5).float()
loss = loss_fn(out['logits'], target)
print('loss:', loss.item())
loss.backward()
print('backward OK')

## §3.3 — Freeze schedule by stage

Stage 1 (mistake detection): freeze ViT-L, base Q-Former, LLM, llm_proj.
Stage 2 (explanation projection): freeze everything; train only `llm_proj`.
Stage 3 (LoRA LLM): freeze `llm_proj`; train LoRA adapters on Llama2.

In [ ]:
from src.train import apply_stage

for stage in (1, 2, 3):
    apply_stage(model, stage)
    n = count_trainable(model)
    print(f'stage {stage} trainable params:', n)

## §4.1 — Detection metrics on dummy predictions

In [ ]:
logits = torch.randn(1000)
tgt = (torch.rand_like(logits) > 0.7).float()
metrics = detection_metrics(logits, tgt)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')

## Next steps

1. Extract per-clip features at 2 fps (§4.6) using BLIP-2's ViT-L + Q-Former and Mediapipe Hands. Save as `<clip_id>/visual.npy` and `<clip_id>/hand.npy`.
2. Configure paths in your launch script. Use `configs/epic_tent_o.yaml` or `configs/holo_assist.yaml`.
3. Run Stage 1: `python -m src.train` (after exposing a CLI entry point) or call `train_stage1(cfg, data_root)`.
4. Hook up Llama2-7B + LoRA via `transformers` + `peft` to enable Stage 2/3 explanation training.
5. Compare F1 against paper Table 1 (HoloAssist) / Table 3 (Epic-tent-O).